In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/clean_house_data.csv")
rppi = pd.read_csv("../data/raw/cso_rppi_index.csv")

In [2]:
rppi = rppi[(rppi["Statistic Label"] == "Residential Property Price Index") &
            (rppi["Type of Residential Property"] == "National - all residential properties")]

In [6]:
rppi = rppi[["TLIST(M1)", "VALUE"]].rename(columns={"TLIST(M1)": "ym", "VALUE": "rppi_index"})

In [3]:
df["date"] = pd.to_datetime(df["date"])

In [4]:
df["ym"] = df["date"].dt.year * 100 + df["date"].dt.month

In [7]:
df = df.merge(rppi, on="ym", how="left")

In [8]:
len(df)
df[["date", "ym", "rppi_index"]].head()

,date,ym,rppi_index
0,2010-01-01,201001,113.2
1,2010-01-03,201001,113.2
2,2010-01-04,201001,113.2
3,2010-01-04,201001,113.2
4,2010-01-04,201001,113.2


In [10]:
latest = rppi.loc[rppi["ym"] == rppi["ym"].max(), "rppi_index"].iloc[0]
print(latest)

210.5


In [11]:
df["coef"] = latest / df["rppi_index"]
df["price"] = df["price"] * df["coef"]
display(df)

,date,address,county,eircode,price,is_new,ym,rppi_index,coef
0,2010-01-01,"5 Braemor Drive, Churchtown, Co.Dublin",Dublin,NaN,637822.438163,False,201001,113.2,1.859541
1,2010-01-03,"134 Ashewood Walk, Summerhill Lane, Portlaoise",Laois,NaN,390457.045053,True,201001,113.2,1.859541
2,2010-01-04,"1 Meadow Avenue, Dundrum, Dublin 14",Dublin,NaN,815408.568905,False,201001,113.2,1.859541
3,2010-01-04,"1 The Haven, Mornington",Meath,NaN,743816.254417,False,201001,113.2,1.859541
4,2010-01-04,"11 Melville Heights, Kilkenny",Kilkenny,NaN,297526.501767,False,201001,113.2,1.859541
...,...,...,...,...,...,...,...,...,...
776606,2026-11-13,"DOORADOYLE RD, DOORADOYLE, LIMERICK",Limerick,V94HN2X,NaN,False,202611,NaN,NaN
776607,2026-11-13,"DRUMAVILLE, MALIN, DONEGAL",Donegal,F93TW90,NaN,False,202611,NaN,NaN
776608,2026-11-13,"GLAS SHEEDY, 8 CLAR AOIBHINN, CLAREMORRIS",Mayo,F12T928,NaN,False,202611,NaN,NaN
776609,2026-11-13,"IONA, 27 NUNS WALK, BALLYPHEHANE",Cork,T12A6Y9,NaN,False,202611,NaN,NaN


In [13]:
df = df[df["rppi_index"].notna()]